# Python `dict` vs Java `ConcurrentHashMap`

Java's [`ConcurrentHashMap`](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/concurrent/ConcurrentHashMap.html) is a thread-safe map with **atomic compound operations** (`putIfAbsent`, `compute`, `merge`, …) and **weakly consistent** iteration.

Python's built-in `dict` (CPython 3.7+) is **thread-safe for individual operations** thanks to the GIL, but **compound check-then-act patterns are not atomic**. Iteration during concurrent writes can raise `RuntimeError`.

The tests below stress both and print a comparison report.

In [1]:
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Callable

# --- ConcurrentHashMap API mapping -----------------------------------------

CHM_FEATURES: dict[str, dict] = {
    "put / get / remove": {
        "java": "Thread-safe single-key ops",
        "python": "d[k]=v, d[k], d.pop(k) — atomic single ops (GIL)",
    },
    "putIfAbsent": {
        "java": "Atomic: insert only if key missing",
        "python_atomic": "d.setdefault(k, v) — one call, atomic under GIL",
        "python_non_atomic": "if k not in d: d[k] = v — race-prone",
    },
    "replace": {
        "java": "Atomic compare-and-swap on value",
        "python": "No built-in; use threading.Lock",
    },
    "compute / computeIfAbsent": {
        "java": "Atomic read-modify-write per key",
        "python": "No built-in atomic compute; manual lock required",
    },
    "size / isEmpty": {
        "java": "Approximate under heavy contention (documented)",
        "python": "len(d) — snapshot, stale mid-update but won't crash",
    },
    "iteration": {
        "java": "Weakly consistent iterator (no ConcurrentModificationException)",
        "python": "for k in d — RuntimeError if dict resized during iteration",
    },
}

print("ConcurrentHashMap ↔ Python dict feature map:\n")
for name, info in CHM_FEATURES.items():
    print(f"  {name}")
    for k, v in info.items():
        print(f"    {k}: {v}")
    print()

ConcurrentHashMap ↔ Python dict feature map:

  put / get / remove
    java: Thread-safe single-key ops
    python: d[k]=v, d[k], d.pop(k) — atomic single ops (GIL)

  putIfAbsent
    java: Atomic: insert only if key missing
    python_atomic: d.setdefault(k, v) — one call, atomic under GIL
    python_non_atomic: if k not in d: d[k] = v — race-prone

  replace
    java: Atomic compare-and-swap on value
    python: No built-in; use threading.Lock

  compute / computeIfAbsent
    java: Atomic read-modify-write per key
    python: No built-in atomic compute; manual lock required

  size / isEmpty
    java: Approximate under heavy contention (documented)
    python: len(d) — snapshot, stale mid-update but won't crash

  iteration
    java: Weakly consistent iterator (no ConcurrentModificationException)
    python: for k in d — RuntimeError if dict resized during iteration



In [2]:
@dataclass
class TestResult:
    name: str
    chm_expectation: str
    passed: bool
    detail: str
    matches_chm: bool  # True if Python dict behaves like ConcurrentHashMap here


results: list[TestResult] = []


def record(name: str, chm_expectation: str, passed: bool, detail: str, matches_chm: bool) -> None:
    r = TestResult(name, chm_expectation, passed, detail, matches_chm)
    results.append(r)
    status = "PASS" if passed else "FAIL"
    chm = "matches CHM" if matches_chm else "differs from CHM"
    print(f"[{status} | {chm}] {name}: {detail}")


def run_threads(workers: int, fn: Callable[[int], None], *, barrier: threading.Barrier | None = None) -> None:
    threads = [threading.Thread(target=fn, args=(i,)) for i in range(workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()


NUM_THREADS = 32
OPS_PER_THREAD = 5_000

In [3]:
# Test 1: Concurrent put (like CHM.put) — distinct keys, no lost writes
# CHM guarantee: every put is visible; final size == number of distinct keys.

d: dict[int, int] = {}
errors: list[str] = []
lock = threading.Lock()


def writer_put(thread_id: int) -> None:
  try:
    base = thread_id * OPS_PER_THREAD
    for i in range(OPS_PER_THREAD):
      d[base + i] = thread_id
  except Exception as exc:
    with lock:
      errors.append(str(exc))


run_threads(NUM_THREADS, writer_put)
expected_size = NUM_THREADS * OPS_PER_THREAD
lost_writes = expected_size - len(d)

record(
    "concurrent put (distinct keys)",
    "All puts visible; size == distinct keys inserted",
    lost_writes == 0 and not errors,
    f"size={len(d)} expected={expected_size} lost={lost_writes} errors={errors[:3]}",
    matches_chm=lost_writes == 0,
)

[PASS | matches CHM] concurrent put (distinct keys): size=160000 expected=160000 lost=0 errors=[]


In [4]:
# Test 2: Concurrent get/put on SAME key (like many threads calling CHM.put(k, v))
# CHM guarantee: no crash; final value is one of the written values.

hot_key = "shared"
d2: dict[str, int] = {}
errors2: list[str] = []
seen_values: set[int] = set()
val_lock = threading.Lock()


def writer_same_key(thread_id: int) -> None:
  try:
    for i in range(OPS_PER_THREAD):
      d2[hot_key] = thread_id * 10_000 + i
      with val_lock:
        seen_values.add(d2[hot_key])
  except Exception as exc:
    with val_lock:
      errors2.append(str(exc))


run_threads(NUM_THREADS, writer_same_key)
final = d2.get(hot_key)

record(
    "concurrent put/get same key",
    "No corruption; final value is one of the writes",
    hot_key in d2 and not errors2 and final in seen_values,
    f"final={final} errors={errors2[:3]}",
    matches_chm=not errors2,
)

[PASS | matches CHM] concurrent put/get same key: final=314999 errors=[]


In [5]:
# Test 3: putIfAbsent — atomic (setdefault) vs non-atomic (check-then-act)
# CHM.putIfAbsent: exactly one initializer per key.

KEYS = 100
barrier = threading.Barrier(NUM_THREADS)


def run_put_if_absent(use_setdefault: bool) -> tuple[int, int, list[str]]:
    d: dict[int, int] = {}
    init_count: dict[int, int] = {}
    count_lock = threading.Lock()
    errors: list[str] = []

    def worker(_tid: int) -> None:
        try:
            for k in range(KEYS):
                barrier.wait()
                if use_setdefault:
                    d.setdefault(k, 0)
                else:
                    if k not in d:
                        time.sleep(0)  # yield GIL between check and set
                        with count_lock:
                            init_count[k] = init_count.get(k, 0) + 1
                        d[k] = 0
                d[k] += 1
        except Exception as exc:
            errors.append(str(exc))

    run_threads(NUM_THREADS, worker)
    wrong_values = sum(1 for k in range(KEYS) if d.get(k) != NUM_THREADS)
    double_inits = sum(1 for k in range(KEYS) if init_count.get(k, 0) > 1)
    return wrong_values, double_inits, errors


wrong_atomic, _, err_atomic = run_put_if_absent(use_setdefault=True)
barrier = threading.Barrier(NUM_THREADS)
wrong_racy, double_racy, err_racy = run_put_if_absent(use_setdefault=False)

record(
    "putIfAbsent via setdefault (atomic)",
    "CHM.putIfAbsent — correct final value per key",
    wrong_atomic == 0 and not err_atomic,
    f"keys_with_wrong_value={wrong_atomic}",
    matches_chm=True,
)

record(
    "putIfAbsent via 'if not in' (non-atomic)",
    "CHM.putIfAbsent — exactly one init per key",
    double_racy == 0 and wrong_racy == 0,
    f"keys_initialized_twice={double_racy} keys_with_wrong_value={wrong_racy}",
    matches_chm=double_racy == 0 and wrong_racy == 0,
)

[PASS | matches CHM] putIfAbsent via setdefault (atomic): keys_with_wrong_value=0
[FAIL | differs from CHM] putIfAbsent via 'if not in' (non-atomic): keys_initialized_twice=100 keys_with_wrong_value=100


In [6]:
# Test 4: compute / increment (CHM.compute / merge)
# CHM: atomic read-modify-write. Python dict: plain += is NOT atomic.

COUNTER_KEY = "count"
INCREMENTS = 2_000
d_atomic: dict[str, int] = {COUNTER_KEY: 0}
d_racy: dict[str, int] = {COUNTER_KEY: 0}
counter_lock = threading.Lock()


def increment_atomic(_tid: int) -> None:
    for _ in range(INCREMENTS):
        with counter_lock:
            d_atomic[COUNTER_KEY] = d_atomic[COUNTER_KEY] + 1


def increment_racy(_tid: int) -> None:
    for _ in range(INCREMENTS):
        current = d_racy[COUNTER_KEY]
        time.sleep(0)  # yield GIL between read and write
        d_racy[COUNTER_KEY] = current + 1


run_threads(NUM_THREADS, increment_atomic)
run_threads(NUM_THREADS, increment_racy)

expected = NUM_THREADS * INCREMENTS
atomic_val = d_atomic[COUNTER_KEY]
racy_val = d_racy[COUNTER_KEY]

record(
    "compute/increment with Lock (CHM-like)",
    "CHM.compute — final count exact",
    atomic_val == expected,
    f"count={atomic_val} expected={expected}",
    matches_chm=True,
)

record(
    "compute/increment without Lock",
    "CHM.compute — final count exact",
    racy_val == expected,
    f"count={racy_val} expected={expected} lost={expected - racy_val}",
    matches_chm=racy_val == expected,
)

[PASS | matches CHM] compute/increment with Lock (CHM-like): count=64000 expected=64000
[FAIL | differs from CHM] compute/increment without Lock: count=2017 expected=64000 lost=61983


In [7]:
# Test 5: Iteration during concurrent modification
# CHM: weakly consistent iterator (no exception). Python dict: RuntimeError.

d_iter: dict[int, int] = {i: i for i in range(50_000)}
iter_errors: list[str] = []
stop = threading.Event()


def mutator(_tid: int) -> None:
    i = 50_000
    while not stop.is_set():
        for _ in range(100):
            d_iter[i] = i
            d_iter.pop(i - 1, None)
            i += 1


def reader(_tid: int) -> None:
    for _ in range(500):
        try:
            for k, v in d_iter.items():  # must iterate, not just list()
                pass
        except RuntimeError as exc:
            iter_errors.append(str(exc))


mutators = [threading.Thread(target=mutator, args=(i,)) for i in range(4)]
readers = [threading.Thread(target=reader, args=(i,)) for i in range(8)]
for t in mutators + readers:
    t.start()
time.sleep(1.5)
stop.set()
for t in mutators + readers:
    t.join()

record(
    "iterate while mutating",
    "CHM iterator never throws ConcurrentModificationException",
    len(iter_errors) == 0,
    f"RuntimeError count={len(iter_errors)} sample={iter_errors[:2]}",
    matches_chm=len(iter_errors) == 0,
)

[FAIL | differs from CHM] iterate while mutating: RuntimeError count=98 sample=['dictionary changed size during iteration', 'dictionary changed size during iteration']


In [ ]:
# Test 6: remove during get (CHM.get never crashes the JVM)
# Python dict: single get/pop should not corrupt internal state.

d_rm: dict[int, int] = {i: i for i in range(5_000)}
rm_errors: list[str] = []
stop_rm = threading.Event()


def remover(_tid: int) -> None:
    k = 0
    while not stop_rm.is_set() and k < 5_000:
        d_rm.pop(k, None)
        k += 1


def getter(_tid: int) -> None:
    for _ in range(5_000):
        try:
            _ = d_rm.get(random.randint(0, 4_999), None)
        except Exception as exc:
            rm_errors.append(type(exc).__name__)


threads = [threading.Thread(target=remover, args=(0,))]
threads += [threading.Thread(target=getter, args=(i,)) for i in range(8)]
for t in threads:
    t.start()
time.sleep(0.5)
stop_rm.set()
for t in threads:
    t.join()

record(
    "concurrent get + remove",
    "CHM get/remove — no corruption or crash",
    not rm_errors,
    f"errors={rm_errors[:3]}",
    matches_chm=not rm_errors,
)

In [ ]:
# Summary

print("=" * 72)
print("SUMMARY: Python dict thread-safety vs ConcurrentHashMap")
print("=" * 72)

matches = [r for r in results if r.matches_chm]
differs = [r for r in results if not r.matches_chm]

print(f"\nBehaves like ConcurrentHashMap ({len(matches)}):")
for r in matches:
    print(f"  ✓ {r.name}")

print(f"\nDiffers from ConcurrentHashMap ({len(differs)}):")
for r in differs:
    print(f"  ✗ {r.name} — {r.detail}")

print("\n--- Takeaways ---")
print("• Single-key put/get/pop/deleteitem: thread-safe (GIL), similar to CHM basic ops.")
print("• setdefault: atomic putIfAbsent equivalent — use this instead of 'if not in'.")
print("• Read-modify-write (+=, compute, merge): NOT atomic — wrap with Lock or use a dedicated concurrent map.")
print("• Iteration during writes: dict raises RuntimeError; CHM uses weakly consistent iterators.")
print("\nClosest stdlib alternative: threading.Lock around compound ops, or concurrent collections from third-party libs.")